In [1]:
from platform import python_version
print(python_version())

3.11.14


In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as npmtd
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

import json
import requests
import pandas as pd

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import MTD
from libs.GDC_lib import GDC
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'
PSI_ID = 'TCGA-PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/config/all_lfc_cutoffs_TCGA-PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=False, verbose=False)
# print("\nEcho Parameters:")
# print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD
>>> Tumor


### GDC - no memory restriction to get all data available

In [5]:
gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [6]:
force=False
verbose=False

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)
prog_list.sort()

"; ".join(prog_list)

'ALCHEMIST; APOLLO; BEATAML1.0; CCDI; CCG; CDDP_EAGLE; CGCI; CMI; CPTAC; CTSP; EXCEPTIONAL_RESPONDERS; FM; HCMI; MATCH; MMRF; MP2PRT; NCICCR; OHSU; ORGANOID; RC; REBC; TARGET; TCGA; TRIO; VAREPOP; WCDT'

In [7]:
PROG_ID = 'TCGA'

df_psi = gdc.get_primary_sites(prog_id=PROG_ID, verbose=verbose)
print(len(df_psi))
df_psi.head(3)

33


,prog_id,gdc_project_id,disease_id,psi_id,primary_site,disease_context,cbioportal_study_id,mapping_status,recommended_mutation_source,notes,alternative_cbioportal_study_ids,review_notes
0,TCGA,TCGA-ACC,ACC,TCGA-ACC,Adrenal gland,Adrenocortical carcinoma,acc_tcga_pan_can_atlas_2018,formulaic_tcga_pan_can_atlas,GDC MAF first; cBioPortal only if mapping is validated,Validate with cBioPortal /api/studies before production.,NaN,NaN
1,TCGA,TCGA-BLCA,BLCA,TCGA-BLCA,Bladder,Bladder urothelial carcinoma,blca_tcga_pan_can_atlas_2018,formulaic_tcga_pan_can_atlas,GDC MAF first; cBioPortal only if mapping is validated,Validate with cBioPortal /api/studies before production.,NaN,NaN
2,TCGA,TCGA-BRCA,BRCA,TCGA-BRCA,Breast,Breast invasive carcinoma,brca_tcga_pan_can_atlas_2018,formulaic_tcga_pan_can_atlas,GDC MAF first; cBioPortal only if mapping is validated,Validate with cBioPortal /api/studies before production.,NaN,NaN


### Open primary cites from cbio

In [8]:
df_psi = gdc.open_primary_sites_cbio(verbose=True)

df_psi.columns

Table opened ((89, 12)) at '/home/flavio/uv/perturb_agent/data/gdc_to_cbioportal_study_mapping.tsv'


Index(['prog_id', 'gdc_project_id', 'disease_id', 'psi_id', 'primary_site', 'disease_context',
       'cbioportal_study_id', 'mapping_status', 'recommended_mutation_source', 'notes',
       'alternative_cbioportal_study_ids', 'review_notes'],
      dtype='object')

In [9]:
df_psi.head(3).T

,0,1,2
prog_id,TCGA,TCGA,TCGA
gdc_project_id,TCGA-ACC,TCGA-BLCA,TCGA-BRCA
disease_id,ACC,BLCA,BRCA
psi_id,TCGA-ACC,TCGA-BLCA,TCGA-BRCA
primary_site,Adrenal gland,Bladder,Breast
disease_context,Adrenocortical carcinoma,Bladder urothelial carcinoma,Breast invasive carcinoma
cbioportal_study_id,acc_tcga_pan_can_atlas_2018,blca_tcga_pan_can_atlas_2018,brca_tcga_pan_can_atlas_2018
mapping_status,formulaic_tcga_pan_can_atlas,formulaic_tcga_pan_can_atlas,formulaic_tcga_pan_can_atlas
recommended_mutation_source,GDC MAF first; cBioPortal only if mapping is validated,GDC MAF first; cBioPortal only if mapping is validated,GDC MAF first; cBioPortal only if mapping is validated
notes,Validate with cBioPortal /api/studies before production.,Validate with cBioPortal /api/studies before production.,Validate with cBioPortal /api/studies before production.


In [10]:
DISEASE_ID = 'ACC'
DISEASE_ID = 'PAAD'

df_psi = df_psi[ (df_psi.disease_id == DISEASE_ID) & (~pd.isnull(df_psi.primary_site)) & (~pd.isnull(df_psi.cbioportal_study_id)) ].copy()
dfa = df_psi.groupby(['prog_id', 'psi_id', 'disease_id', 'primary_site', 'gdc_project_id', 'cbioportal_study_id']).size().reset_index()

dfa[ ['prog_id', 'psi_id', 'disease_id', 'primary_site', 'gdc_project_id', 'cbioportal_study_id'] ]

,prog_id,psi_id,disease_id,primary_site,gdc_project_id,cbioportal_study_id
0,CCLE,CCLE-PAAD,PAAD,Pancreas,CCLE-PAAD,ccle_broad_2019
1,CPTAC,CPTAC-PAAD,PAAD,Pancreas,CPTAC-3,paad_cptac_2021
2,CPTAC,CPTAC-PAAD_GDC,PAAD,Pancreas,CPTAC-3,pancreas_cptac_gdc
3,TCGA,TCGA-PAAD,PAAD,Pancreas,TCGA-PAAD,paad_tcga_pan_can_atlas_2018


In [11]:
prog_id = 'CCLE'  # DepMap 
psi_id = 'CCLE-BRCA'

prog_id = 'CPTAC'
psi_id = 'CPTAC-BRCA'

df_psi = gdc.get_primary_sites(prog_id=prog_id, verbose=verbose)
df_psi.head(2)

,prog_id,gdc_project_id,disease_id,psi_id,primary_site,disease_context,cbioportal_study_id,mapping_status,recommended_mutation_source,notes,alternative_cbioportal_study_ids,review_notes
0,CPTAC,CPTAC-2,BRCA,CPTAC-BRCA,Breast,Breast cancer context,brca_cptac_2020,reviewed_candidate_cptac,GDC MAF first; cBioPortal only if mapping is validated,CPTAC breast may not map to one simple public mutation study.,breast_cptac_gdc,cBioPortal has BRCA CPTAC 2020 and Breast CPTAC GDC 2025; validate which mat...
1,CPTAC,CPTAC-2,COAD,CPTAC-COAD,Colon / Rectum,Colon cancer / colorectal adenocarcinoma,coad_cptac_2019,curated_context_mapping,GDC MAF first; cBioPortal only if mapping is validated,Use only for colon/rectum context inside CPTAC-2.,NaN,NaN


In [12]:

gdc.set_primary_site(psi_id)

True

In [13]:
gdc.gdc_project_id, gdc.primary_site

('CPTAC-2', 'Breast')

In [20]:
force=False
verbose=True

df_cases, df_subt, _ = gdc.get_cases_and_subtypes(batch_size=200, do_filter=True, force=force, verbose=verbose)

print(df_cases.shape)
print(df_cases.primary_site.unique())
df_cases.head(3)

Table opened ((134, 26)) at '/home/flavio/uv/perturb_agent/data/CPTAC/CPTAC-BRCA/cases_for_CPTAC-BRCA.tsv'
(17, 26)
['Breast']


,primary_site,disease_type,case_id,diagnoses,gdc_project_id,subtype_global,stage_ajcc,primary_diagnosis,tumor_grade,stage_clin,...,primary_site_norm,disease_type_norm,diagnosis_norm,tumor_class,histology,subtype_tissue,consistency,validity,n,frac
0,Breast,Ductal and Lobular Neoplasms,c0cef5fc-0c80-4812-aece-97c351747f26,"[{'figo_stage': 'Not Reported', 'primary_diagnosis': 'Infiltrating lobular c...",CPTAC-2,lobular,unknown,"Infiltrating lobular carcinoma, NOS",Not Reported,NaN,...,breast,ductal and lobular neoplasms,infiltrating lobular carcinoma,other,epithelial,lobular,ok,valid,1,0.007
1,Breast,Ductal and Lobular Neoplasms,e2549d98-1b68-4c4d-b7f6-311fdd05b260,"[{'figo_stage': 'Not Reported', 'primary_diagnosis': 'Infiltrating lobular c...",CPTAC-2,lobular,unknown,"Infiltrating lobular carcinoma, NOS",Not Reported,NaN,...,breast,ductal and lobular neoplasms,infiltrating lobular carcinoma,other,epithelial,lobular,ok,valid,1,0.007
2,Breast,Ductal and Lobular Neoplasms,aca05259-4b08-413b-a118-9520ccc5a3b4,"[{'figo_stage': 'Not Reported', 'primary_diagnosis': 'Infiltrating lobular c...",CPTAC-2,lobular,unknown,"Infiltrating lobular carcinoma, NOS",Not Reported,NaN,...,breast,ductal and lobular neoplasms,infiltrating lobular carcinoma,other,epithelial,lobular,ok,valid,1,0.007


In [21]:
df_subt

,psi_id,subtype_global,tumor_class,subtype_tissue,stage,n
0,CPTAC-BRCA,lobular,other,lobular,unknown,14
1,CPTAC-BRCA,adenocarcinoma_generic,adenocarcinoma,adenocarcinoma_generic,unknown,1
2,CPTAC-BRCA,ductal,other,ductal,unknown,1
3,CPTAC-BRCA,squamous,squamous_cell_carcinoma,squamous,unknown,1


In [22]:
for isubt, row in df_subt.iterrows():
    subtype_global = row.subtype_global
    tumor_class = row.tumor_class
    subtype_tissue = row.subtype_tissue

    df_samples = gdc.get_samples_for_subtypes(
        subtype_global=subtype_global,
        tumor_class=tumor_class,
        subtype_tissue=subtype_tissue,
        batch_size=200,
        force=False,
        verbose=verbose,
    )
    print(f"{isubt}) {gdc.s_case}")

    if df_samples.empty:
        print(f"No samples found for PSI_ID: {psi_id} subtype: {subtype_global} tumor_class: {tumor_class} subtype_tissue: {subtype_tissue}")
    else:
        print(f"There are {len(df_samples)} samples for PSI_ID: {psi_id} subtype: {subtype_global} tumor_class: {tumor_class} subtype_tissue: {subtype_tissue}")


Table opened ((134, 26)) at '/home/flavio/uv/perturb_agent/data/CPTAC/CPTAC-BRCA/cases_for_CPTAC-BRCA.tsv'
Table opened ((576, 14)) at '/home/flavio/uv/perturb_agent/data/CPTAC/CPTAC-BRCA/samples/samples_for_CPTAC-2_BRCA_Breast_subtype_lobular_tumor_other_tissue_lobular.tsv'
0) CPTAC-2_BRCA_Breast_subtype_lobular_tumor_other_tissue_lobular
There are 576 samples for PSI_ID: CPTAC-BRCA subtype: lobular tumor_class: other subtype_tissue: lobular
Table opened ((134, 26)) at '/home/flavio/uv/perturb_agent/data/CPTAC/CPTAC-BRCA/cases_for_CPTAC-BRCA.tsv'
Table opened ((10, 14)) at '/home/flavio/uv/perturb_agent/data/CPTAC/CPTAC-BRCA/samples/samples_for_CPTAC-2_BRCA_Breast_subtype_adenocarcinoma-generic_tumor_adenocarcinoma_tissue_adenocarcinoma-generic.tsv'
1) CPTAC-2_BRCA_Breast_subtype_adenocarcinoma-generic_tumor_adenocarcinoma_tissue_adenocarcinoma-generic
There are 10 samples for PSI_ID: CPTAC-BRCA subtype: adenocarcinoma_generic tumor_class: adenocarcinoma subtype_tissue: adenocarcinoma

### Clinical Supplements

In [23]:
df_clin = gdc.get_gdc_clinical_data(case_ids=df_samples.case_id, batch_size=50)
print(df_clin.shape)

df_clin.head(3)

(17, 37)


,case_id,barcode_case,project_id,primary_site,disease_type,sex_at_birth,gender,race,ethnicity,vital_status,...,tumor_stage,ajcc_pathologic_stage,ajcc_pathologic_t,ajcc_pathologic_n,ajcc_pathologic_m,ajcc_clinical_stage,ajcc_clinical_t,ajcc_clinical_n,ajcc_clinical_m,classification_of_tumor
0,2ab368b4-f003-4781-866e-52f6a4e65a76,03BR005,CPTAC-2,Breast,Ductal and Lobular Neoplasms,female,NaN,white,not hispanic or latino,Not Reported,...,NaN,Stage IIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not reported
1,2af3bcd7-505f-4352-9862-8f5f45aa3e63,05BR026,CPTAC-2,Breast,Ductal and Lobular Neoplasms,female,NaN,white,not hispanic or latino,Not Reported,...,NaN,Stage IIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not reported
2,35920ef3-51ee-47f7-af63-2e278cc33951,20BR008,CPTAC-2,Breast,Ductal and Lobular Neoplasms,female,NaN,black or african american,not hispanic or latino,Not Reported,...,NaN,Stage IIIB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not reported


In [24]:
def get_gdc_case_by_sample_barcode(
    barcode_sample: str,
    project_id: str | None = None,
) -> pd.DataFrame:
    endpoint = "https://api.gdc.cancer.gov/cases"

    conditions = [
        {
            "op": "in",
            "content": {
                "field": "samples.submitter_id",
                "value": [barcode_sample],
            },
        }
    ]

    if project_id:
        conditions.append(
            {
                "op": "in",
                "content": {
                    "field": "project.project_id",
                    "value": [project_id],
                },
            }
        )

    filters = (
        conditions[0]
        if len(conditions) == 1
        else {
            "op": "and",
            "content": conditions,
        }
    )

    params = {
        "filters": json.dumps(filters),
        "fields": ",".join(
            [
                "case_id",
                "submitter_id",
                "project.project_id",
                "demographic.age_at_index",
                "demographic.days_to_birth",
                "diagnoses.age_at_diagnosis",
                "samples.sample_id",
                "samples.submitter_id",
                "samples.sample_type",
                "samples.days_to_collection",
            ]
        ),
        "format": "JSON",
        "size": 100,
    }

    response = requests.get(
        endpoint,
        params=params,
        timeout=120,
    )
    response.raise_for_status()

    return pd.json_normalize(
        response.json().get("data", {}).get("hits", [])
    )


def get_gdc_sample_by_barcode(
    barcode_sample: str,
    project_id: str | None = None,
) -> pd.DataFrame:
    df_samp = get_gdc_case_by_sample_barcode(
        barcode_sample=barcode_sample,
        project_id=project_id,
    )

    records = []

    for _, row_case in df_samp.iterrows():
        for sample in row_case.get("samples", []) or []:
            if sample.get("submitter_id") != barcode_sample:
                continue

            records.append(
                {
                    "case_id": row_case.get("case_id"),
                    "case_submitter_id": row_case.get(
                        "submitter_id"
                    ),
                    "project_id": row_case.get(
                        "project.project_id"
                    ),
                    "sample_id": sample.get("sample_id"),
                    "sample_submitter_id": sample.get(
                        "submitter_id"
                    ),
                    "sample_type": sample.get(
                        "sample_type"
                    ),
                    "days_to_collection": sample.get(
                        "days_to_collection"
                    ),
                    "days_to_sample_procurement": sample.get(
                        "days_to_sample_procurement"
                    ),
                }
            )

    return pd.DataFrame(records)
    

In [ ]:
for isubt, row in df_subt.iterrows():
    subtype_global = row.subtype_global
    tumor_class = row.tumor_class
    subtype_tissue = row.subtype_tissue

    df_samples = gdc.get_samples_for_subtypes(
        subtype_global=subtype_global,
        tumor_class=tumor_class,
        subtype_tissue=subtype_tissue,
        batch_size=200,
        force=False,
        verbose=verbose,
    )
    print(f"{isubt}) {gdc.s_case}")


    dic = {}

    if df_samples.empty:
        print(f"No samples found for PSI_ID: {psi_id} subtype: {subtype_global} tumor_class: {tumor_class} subtype_tissue: {subtype_tissue}")
    else:
        print(f"There are {len(df_samples)} samples for PSI_ID: {psi_id} subtype: {subtype_global} tumor_class: {tumor_class} subtype_tissue: {subtype_tissue}")

        for irow, row in df_samples.iterrows():

            if row.case_id in dic.keys():
                continue
            else:
                print(irow, row.case_id, row.barcode_sample)

                dfa = get_gdc_case_by_sample_barcode(row.barcode_sample)

                try:
                    age_at_diagnosis = np.round(dfa.diagnoses[0][0]['age_at_diagnosis'] / 365.25, 2)
                    dic[row.case_id] = age_at_diagnosis
                except:
                    age_at_diagnosis = -1
                
            print(f"Age at diagnosis: {irow} {age_at_diagnosis}")
            print("")

    break


Table opened ((134, 26)) at '/home/flavio/uv/perturb_agent/data/CPTAC/CPTAC-BRCA/cases_for_CPTAC-BRCA.tsv'
Table opened ((576, 14)) at '/home/flavio/uv/perturb_agent/data/CPTAC/CPTAC-BRCA/samples/samples_for_CPTAC-2_BRCA_Breast_subtype_lobular_tumor_other_tissue_lobular.tsv'
0) CPTAC-2_BRCA_Breast_subtype_lobular_tumor_other_tissue_lobular
There are 576 samples for PSI_ID: CPTAC-BRCA subtype: lobular tumor_class: other subtype_tissue: lobular
0 e2549d98-1b68-4c4d-b7f6-311fdd05b260 99df02e0-56b3-4b00-80b7-ea58ce
Age at diagnosis: 0 65.95

1 e2549d98-1b68-4c4d-b7f6-311fdd05b260 ca14c6dd-2069-4bc5-b1f2-6f2297
2 e2549d98-1b68-4c4d-b7f6-311fdd05b260 ca14c6dd-2069-4bc5-b1f2-6f2297
3 e2549d98-1b68-4c4d-b7f6-311fdd05b260 99df02e0-56b3-4b00-80b7-ea58ce
4 e2549d98-1b68-4c4d-b7f6-311fdd05b260 ca14c6dd-2069-4bc5-b1f2-6f2297
5 e2549d98-1b68-4c4d-b7f6-311fdd05b260 ca14c6dd-2069-4bc5-b1f2-6f2297
6 e2549d98-1b68-4c4d-b7f6-311fdd05b260 ca14c6dd-2069-4bc5-b1f2-6f2297
7 e2549d98-1b68-4c4d-b7f6-311fdd05b2

KeyboardInterrupt: 

In [27]:
dfa = get_gdc_case_by_sample_barcode(row.barcode_sample)
age_at_diagnosis = np.round(dfa.diagnoses[0][0]['age_at_diagnosis'] / 365.25, 2)
age_at_diagnosis

65.95

In [30]:
df_samples.columns

Index(['case_id', 'submitter_id', 'sample_id', 'sample_type', 'barcode_sample', 'file_id',
       'file_name', 'data_type', 'data_format', 'psi_id', 'subtype_global', 'tumor_class',
       'subtype_tissue', 'stage'],
      dtype='object')

In [31]:
df_samples[ ['case_id', 'barcode_sample'] ]

,case_id,barcode_sample
0,e2549d98-1b68-4c4d-b7f6-311fdd05b260,99df02e0-56b3-4b00-80b7-ea58ce
1,e2549d98-1b68-4c4d-b7f6-311fdd05b260,ca14c6dd-2069-4bc5-b1f2-6f2297
2,e2549d98-1b68-4c4d-b7f6-311fdd05b260,ca14c6dd-2069-4bc5-b1f2-6f2297
3,e2549d98-1b68-4c4d-b7f6-311fdd05b260,99df02e0-56b3-4b00-80b7-ea58ce
4,e2549d98-1b68-4c4d-b7f6-311fdd05b260,ca14c6dd-2069-4bc5-b1f2-6f2297
...,...,...
571,007df8da-3d8c-497d-b7d3-3a02596f9577,7604cab5-d108-4fbd-8bc7-2bee52
572,007df8da-3d8c-497d-b7d3-3a02596f9577,7604cab5-d108-4fbd-8bc7-2bee52
573,007df8da-3d8c-497d-b7d3-3a02596f9577,7604cab5-d108-4fbd-8bc7-2bee52
574,007df8da-3d8c-497d-b7d3-3a02596f9577,7604cab5-d108-4fbd-8bc7-2bee52


In [ ]:
row = df_samples.iloc[0]
row.barcode_sample

In [ ]:
dfa = get_gdc_case_by_sample_barcode(row.barcode_sample)
dfa.T

In [ ]:
age_at_diagnosis = np.round(dfa.diagnoses[0][0]['age_at_diagnosis'] / 365.25, 2)
age_at_diagnosis

In [ ]:

    barcode_sample: str,

In [ ]:
df_supp[
        ["data_category", "data_type", "data_format"]
    ].value_counts().reset_index(name="n")

In [ ]:
df_cases.gdc_project_id.unique()

In [ ]:
df_cases.primary_site.unique()

### Demographic data

In [ ]:
df_cases.head(2)

In [ ]:
df_demo = get_gdc_clinical_data(case_ids=df_cases.case_id, batch_size=50)

In [ ]:
print(len(df_demo), len(df_cases))
df_demo.head(4).T

In [ ]:
gdc.root_disease

In [ ]:
fname = 'demographics_for_%s.tsv'
fname = fname % gdc.psi_id
pdwritecsv(df_demo, fname, gdc.root_disease)

In [ ]:
import requests, json

def validate_gdc_project_id(
    project_id: str,
    base_url: str = "https://api.gdc.cancer.gov",
) -> dict | None:
    filters = {
        "op": "in",
        "content": {
            "field": "project_id",
            "value": [project_id],
        },
    }

    params = {
        "filters": json.dumps(filters),
        "fields": "project_id,name,program.name,primary_site,disease_type",
        "format": "JSON",
        "size": 1,
    }

    response = requests.get(
        f"{base_url}/projects",
        params=params,
        timeout=60,
    )
    response.raise_for_status()

    hits = response.json().get("data", {}).get("hits", [])
    return hits[0] if hits else None

validate_gdc_project_id(gdc.gdc_project_id)

In [ ]:
import requests, json
batch_size=200


# -------------------------- batch loop ---------------------------
filters = {
    "op": "in",
    "content": {
        "field": "cases.project.project_id",
        "value": [gdc.gdc_project_id],
    },
}
        
all_hits = []
from_ = 0
size_ = batch_size
total = None

gdc.df_cases = pd.DataFrame()
gdc.df_subt = pd.DataFrame()
gdc.df_prof = pd.DataFrame()

df_cases = pd.DataFrame()

try:
    while True:
        print(".", end="")

        params = {
            "filters": json.dumps(filters),
            "fields": ",".join(
                [
                    "case_id",
                    "project.project_id",
                    "primary_site",
                    "disease_type",
                    "diagnoses.primary_diagnosis",
                    "diagnoses.tumor_grade",
                    "diagnoses.ajcc_pathologic_stage",
                    "diagnoses.ajcc_clinical_stagediagnoses.figo_stage",
                    "diagnoses.tumor_stage",
                ]
            ),
            "format": "JSON",
            "size": size_,
            "from": from_,
        }

        res = requests.get(gdc.url_gdc_cases, params=params)
        response = res.json()

        if "data" not in response.keys():
            print(f"No data found while searching for '{gdc.psi_or_gdc_project_id}'")
            print(">>> response", response)
            # return gdc.df_cases, gdc.df_subt, gdc.df_prof

        hits = response.get("data", {}).get("hits", [])

        if total is None:
            total = response["data"]["pagination"]["total"]

        if not hits:
            break

        all_hits.extend(hits)
        from_ += size_

    print("\n")

    if all_hits == []:
        print(f"No subtypes found for {gdc.psi_or_gdc_project_id} - filter value: {gdc.gdc_project_id}")
        # return gdc.df_cases, gdc.df_subt, gdc.df_prof
except:
    print("error")

In [ ]:
            for isubt, row in df_subt.iterrows():
                subtype_global = row.subtype_global
                tumor_class = row.tumor_class
                subtype_tissue = row.subtype_tissue

                df_samples = gdc.get_samples_for_subtypes(
                    subtype_global=subtype_global,
                    tumor_class=tumor_class,
                    subtype_tissue=subtype_tissue,
                    batch_size=200,
                    force=False,
                    verbose=verbose,
                )

In [ ]:
gdc.gdc_project_id

In [ ]:
verbose=False
force=False

prog_id_list = ['TCGA', 'CPTAC', 'CCLE', 'TARGET', '']
prog_id_list = ['CCLE']

for i, prog_id in enumerate(prog_id_list):

    print(f"{i}) prog_id {prog_id}")

    # df_cases, df_all_samples, df_all_mutations = gdc.loop_program_psi_get_cases_samples_mut(prog_id=prog_id, force=force, verbose=verbose)

    print("")
    break

print("\n--------------- end ---------------")


In [ ]:
df_all_samples.head(3)

In [ ]:
import json
import requests
import pandas as pd
import numpy as np

GDC_CASES_ENDPOINT = gdc.url_gdc_cases


def get_gdc_clinical_data(
    case_ids: pd.Series,
    batch_size: int = 200,
) -> pd.DataFrame:
    """
    Retrieve case-level demographic and diagnosis data from the GDC.

    Returns one row per case-diagnosis combination because a case may
    contain more than one diagnosis.
    """

    case_ids = np.unique(case_ids)

    fields = [
        # Case
        "case_id",
        "submitter_id",
        "project.project_id",
        "primary_site",
        "disease_type",

        # Demographic
        "demographic.demographic_id",
        "demographic.sex_at_birth",
        # "demographic.gender",   deprecated in newer dictionary versions
        "demographic.race",
        "demographic.ethnicity",
        "demographic.vital_status",
        "demographic.days_to_birth",
        "demographic.days_to_death",
        "demographic.year_of_birth",
        "demographic.year_of_death",

        # Diagnosis
        "diagnoses.diagnosis_id",
        "diagnoses.submitter_id",
        "diagnoses.primary_diagnosis",
        "diagnoses.morphology",
        "diagnoses.tissue_or_organ_of_origin",
        "diagnoses.site_of_resection_or_biopsy",
        "diagnoses.age_at_diagnosis",
        "diagnoses.days_to_diagnosis",
        "diagnoses.days_to_last_follow_up",
        "diagnoses.days_to_last_known_disease_status",
        "diagnoses.last_known_disease_status",
        "diagnoses.progression_or_recurrence",
        "diagnoses.tumor_grade",
        "diagnoses.tumor_stage",
        "diagnoses.ajcc_pathologic_stage",
        "diagnoses.ajcc_pathologic_t",
        "diagnoses.ajcc_pathologic_n",
        "diagnoses.ajcc_pathologic_m",
        "diagnoses.ajcc_clinical_stage",
        "diagnoses.ajcc_clinical_t",
        "diagnoses.ajcc_clinical_n",
        "diagnoses.ajcc_clinical_m",
        "diagnoses.classification_of_tumor",
    ]

    records = []
    icount=-1

    for start in range(0, len(case_ids), batch_size):
        batch = case_ids[start:start + batch_size]

        if isinstance(batch, np.ndarray):
            batch = batch.tolist()

        filters = {
            "op": "in",
            "content": {
                "field": "case_id",
                "value": batch,
            },
        }

        params = {
            "filters": json.dumps(filters),
            "fields": ",".join(fields),
            "format": "JSON",
            "size": len(batch),
        }

        response = requests.get(
            GDC_CASES_ENDPOINT,
            params=params,
            timeout=120,
        )
        response.raise_for_status()

        hits = response.json()["data"]["hits"]

        for case in hits:

            icount += 1
            if icount%10==0:
                print(".", end='')

            demographic = case.get("demographic") or {}
            project = case.get("project") or {}
            diagnoses = case.get("diagnoses") or []

            # Preserve cases for which diagnosis information is unavailable.
            if not diagnoses:
                diagnoses = [{}]

            for diagnosis in diagnoses:
                records.append(
                    {
                        # Case identifiers
                        "case_id": case.get("case_id"),
                        "barcode_case": case.get("submitter_id"),
                        "project_id": project.get("project_id"),
                        "primary_site": case.get("primary_site"),
                        "disease_type": case.get("disease_type"),

                        # Demographic
                        "sex_at_birth": demographic.get("sex_at_birth"),
                        "gender": demographic.get("gender"),
                        "race": demographic.get("race"),
                        "ethnicity": demographic.get("ethnicity"),
                        "vital_status": demographic.get("vital_status"),
                        "days_to_birth": demographic.get("days_to_birth"),
                        "days_to_death": demographic.get("days_to_death"),
                        "year_of_birth": demographic.get("year_of_birth"),
                        "year_of_death": demographic.get("year_of_death"),

                        # Diagnosis
                        "diagnosis_id": diagnosis.get("diagnosis_id"),
                        "barcode_diagnosis": diagnosis.get("submitter_id"),
                        "primary_diagnosis": diagnosis.get(
                            "primary_diagnosis"
                        ),
                        "morphology": diagnosis.get("morphology"),
                        "tissue_or_organ_of_origin": diagnosis.get(
                            "tissue_or_organ_of_origin"
                        ),
                        "site_of_resection_or_biopsy": diagnosis.get(
                            "site_of_resection_or_biopsy"
                        ),
                        "age_at_diagnosis": diagnosis.get(
                            "age_at_diagnosis"
                        ),
                        "days_to_diagnosis": diagnosis.get(
                            "days_to_diagnosis"
                        ),
                        "days_to_last_follow_up": diagnosis.get(
                            "days_to_last_follow_up"
                        ),
                        "days_to_last_known_disease_status":
                            diagnosis.get(
                                "days_to_last_known_disease_status"
                            ),
                        "last_known_disease_status": diagnosis.get(
                            "last_known_disease_status"
                        ),
                        "progression_or_recurrence": diagnosis.get(
                            "progression_or_recurrence"
                        ),
                        "tumor_grade": diagnosis.get("tumor_grade"),
                        "tumor_stage": diagnosis.get("tumor_stage"),
                        "ajcc_pathologic_stage": diagnosis.get(
                            "ajcc_pathologic_stage"
                        ),
                        "ajcc_pathologic_t": diagnosis.get(
                            "ajcc_pathologic_t"
                        ),
                        "ajcc_pathologic_n": diagnosis.get(
                            "ajcc_pathologic_n"
                        ),
                        "ajcc_pathologic_m": diagnosis.get(
                            "ajcc_pathologic_m"
                        ),
                        "ajcc_clinical_stage": diagnosis.get(
                            "ajcc_clinical_stage"
                        ),
                        "ajcc_clinical_t": diagnosis.get(
                            "ajcc_clinical_t"
                        ),
                        "ajcc_clinical_n": diagnosis.get(
                            "ajcc_clinical_n"
                        ),
                        "ajcc_clinical_m": diagnosis.get(
                            "ajcc_clinical_m"
                        ),
                        "classification_of_tumor": diagnosis.get(
                            "classification_of_tumor"
                        ),
                    }
                )

    print("\n--------------- end ------------")
    if records == []:
        df = pd.DataFrame()
    else:
        try:
            df = pd.DataFrame(records)
        except Exception as e:
            print(f"Error: {e}")
            df = pd.DataFrame()

    return df